# Solutions · 05 · Departure functions and fugacity

Worked solutions to the exercises of [notebook 05](../notebooks/05_departure_functions_and_fugacity.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, integrate
from engthermo import eos, idealgas, vapor
from engthermo.constants import R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Methane is expanded in a turbo-expander from 60 bar and 250 K to 5 bar (isentropic efficiency 85 %). Find the outlet temperature and the work with the PR equation, and with the ideal-gas model. Which one predicts liquid formation? (Compare the outlet temperature with methane's saturation temperature at 5 bar from `vapor.antoine_Tsat`.)

In [2]:
T1, p1, p2, eta = 250.0, 60e5, 5e5, 0.85
h = lambda T, p, ph: idealgas.enthalpy("methane", T) + eos.solve("pr", "methane", T, p, ph)["H_R"]
s = lambda T, p, ph: idealgas.entropy("methane", T, p) + eos.solve("pr", "methane", T, p, ph)["S_R"]
T_sat = optimize.brentq(lambda T: vapor.eos_psat("pr", "methane", T)["p_sat"] - p2, 100.0, 180.0)      # dew point at 5 bar (PR)
T2s = optimize.brentq(lambda T: s(T, p2, "vapor") - s(T1, p1, "stable"), T_sat, T1)                    # vapour branch
w = eta * (h(T1, p1, "stable") - h(T2s, p2, "vapor"))
T2 = optimize.brentq(lambda T: h(T1, p1, "stable") - h(T, p2, "vapor") - w, T_sat, T1)
T2_ig = idealgas.isentropic_T("methane", T1, p1, p2)
print(f"PR: isentropic outlet {T2s:.1f} K, actual (85 %) outlet {T2:.1f} K, work {w/1e3:.2f} kJ/mol; dew point at 5 bar {T_sat:.1f} K")
print(f"ideal gas: isentropic outlet {T2_ig:.1f} K, {T2_ig - T_sat:.0f} K above the dew point")

PR: isentropic outlet 135.8 K, actual (85 %) outlet 145.0 K, work 2.62 kJ/mol; dew point at 5 bar 135.2 K
ideal gas: isentropic outlet 144.9 K, 10 K above the dew point


Neither model predicts liquid for this pressure ratio, but they differ by nine kelvin, and that is the
whole story: the real-gas outlet is within a fraction of a kelvin of the dew point, the ideal-gas outlet
comfortably above it. A slightly larger pressure ratio, or a colder inlet, and the real expander produces
liquid where the ideal-gas model would still see dry gas - which is exactly how turbo-expanders are used in
gas plants, to condense the heavier components. The difference comes from the enthalpy and entropy
departures at the cold, dense inlet, where methane is far from ideal.

## Exercise 2

Compute the Joule-Thomson temperature drop of CO2 throttled from 60 bar and 320 K to 1 bar (constant enthalpy, PR departures plus ideal-gas enthalpy). Why is this used in some CO2 capture and liquefaction processes?

In [3]:
h_in = idealgas.enthalpy("CO2", 320.0) + eos.solve("pr", "CO2", 320.0, 60e5, "stable")["H_R"]
T_out = optimize.brentq(lambda T: idealgas.enthalpy("CO2", T) + eos.solve("pr", "CO2", T, 1e5)["H_R"] - h_in, 150.0, 320.0)
print(f"Joule-Thomson expansion 60 bar -> 1 bar from 320 K: outlet {T_out:.1f} K, a drop of {320 - T_out:.1f} K")

Joule-Thomson expansion 60 bar -> 1 bar from 320 K: outlet 242.9 K, a drop of 77.1 K


Throttling a dense gas cools it by tens of kelvin - the Joule-Thomson effect, absent in an ideal gas. Cascaded
with heat exchangers it is the basis of Linde-type liquefaction and of some CO$_2$ liquefaction and capture
schemes, where the pressure energy of the compressed gas is spent on cooling itself.

## Exercise 3

**Implement it yourself:** compute $\ln\phi$ of CO2 at 320 K and 50 bar by numerical integration of $\int_0^p (Z - 1)\,dp/p$ and compare with the PR fugacity coefficient.

In [4]:
T, p = 320.0, 50e5
integral = integrate.quad(lambda pp: (eos.solve("pr", "CO2", T, pp, "stable")["Z"] - 1) / pp, 1.0, p, limit=200)[0]
print(f"ln phi by integration of (Z - 1)/p: {integral:.5f};  PR closed form: {eos.solve('pr', 'CO2', T, p)['lnphi']:.5f}")

ln phi by integration of (Z - 1)/p: -0.23129;  PR closed form: -0.23129


$\ln\phi = \int_0^p (Z - 1)\,dp/p$ is the definition of the fugacity coefficient made numerical; the closed-form
expression of the cubic equation is this integral evaluated analytically.